In [ ]:
from __future__ import annotations

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                          TRATAMENTO DE OUTLIERS                              ║
# ║        Remoção de Colunas Constantes com FIT Anti-Leakage por Row Group      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Versão      : 2.1
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de tratamento de outliers (Step 7) da
#  arquitetura de dados do TCC. Nesta versão, o "tratamento" consiste na
#  identificação e remoção de colunas constantes (sem poder discriminativo
#  para o modelo) — detectadas exclusivamente no conjunto de treino (FIT) e
#  removidas de forma idêntica do treino e do teste (TRANSFORM), preservando
#  a garantia de que o teste nunca influencia quais colunas são descartadas.
#
#  O script recebe a estrutura de Parquets imputados gerada na etapa anterior
#  (Step 6 — tratamento de faltantes) e produz a mesma estrutura espelhada,
#  agora sem as colunas constantes identificadas.
#
#  Ao final da execução, o script gera:
#    (a) Parquets "limpos" de treino e teste, por cenário e estratégia;
#    (b) um JSON com a lista de colunas constantes identificadas no treino
#        (auditável, sem dados do teste);
#    (c) relatórios CSV de remoção de constantes por cenário e consolidado.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES GLOBAIS"):
#
#    SCENARIOS          : cenário(s) herdado(s) da etapa de faltantes (padrão:
#                          cenario_A_todos_anos)
#    STRATEGIES         : estratégia(s) de imputação herdadas da etapa
#                          anterior (padrão: HYBRID_AGGRESSIVE)
#    DEFAULT_BATCH_SIZE : nº de linhas por lote no TRANSFORM
#    BASE_PATH          : caminho raiz de armazenamento
#                          (Google Drive: /content/drive/MyDrive/TCC_PLD_Project)
#
#  Arquivos de entrada esperados (gerados pela etapa de tratamento de
#  faltantes — Step 6):
#    .../6_tratamento_de_faltantes/<cenario>/treino/<ESTRATEGIA>/TREINO_IMPUTADO.parquet
#    .../6_tratamento_de_faltantes/<cenario>/teste/<ESTRATEGIA>/TESTE_IMPUTADO.parquet
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Parquets sem constantes →  .../7_tratamento_outliers/<cenario>/
#     {treino,teste}/<ESTRATEGIA>/{TREINO,TESTE}_CLEAN.parquet
#     (compressão Snappy, mesmas linhas, colunas constantes removidas)
#
#  2. Lista de constantes (JSON) →  .../7_tratamento_outliers/<cenario>/
#     estatisticas_fit/<ESTRATEGIA>/constant_cols.json
#     Contém a lista de colunas removidas e metadados do FIT (colunas totais,
#     linhas do treino, número de row groups).
#
#  3. Relatórios (CSV)  →  .../7_tratamento_outliers/<cenario>/relatorios/
#     relatorio_outliers.csv
#     .../7_tratamento_outliers/relatorio_consolidado_outliers.csv
#     Ambos com linhas, colunas removidas/mantidas, tempo de execução e
#     tamanho de saída por split/estratégia/cenário.
#
#  4. Saída no terminal (Rich):
#     • Painéis de FIT e TRANSFORM por cenário/estratégia
#     • Tabela com o resultado do FIT de constantes (totais, removidas, %)
#     • Tabela consolidada final com status OK/ERRO por execução
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ARQUITETURA ANTI-LEAKAGE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  As colunas constantes são identificadas APENAS no TREINO (fit) e a mesma
#  lista é aplicada ao TESTE (transform). O TESTE nunca influencia quais
#  colunas são removidas.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_PATH.
#     Depende da estrutura de Parquets imputados gerada pela etapa anterior
#     (Step 6 — tratamento de faltantes), para o mesmo cenário e estratégia.
#     Este pipeline é local e não faz requisições HTTP externas.
#
#  2. OTIMIZAÇÃO DO FIT DE CONSTANTES (scan por row group)
#     Versão anterior: uma leitura por coluna (pq.read_table(col)), o que
#     exigia uma abertura do arquivo por coluna do catálogo — um custo alto
#     em datasets com milhares de variáveis.
#     Versão atual: o FIT itera por ROW GROUP do Parquet (tipicamente
#     poucas dezenas de passagens), lendo a cada passagem apenas as colunas
#     ainda em rastreio.
#
#  3. EARLY-EXIT POR COLUNA
#     Assim que uma coluna acumula mais de um valor único, ela sai do
#     conjunto de rastreio (não é mais lida nos row groups seguintes). Isso
#     faz o custo de RAM e I/O cair progressivamente ao longo do scan, já
#     que colunas não constantes tendem a ser eliminadas do rastreio logo
#     nos primeiros row groups.
#
#  4. NUNCA MATERIALIZA O ARQUIVO INTEIRO
#     Cada row group é lido, processado (contagem de valores únicos) e
#     descartado antes do próximo, evitando carregar o Parquet completo em
#     memória mesmo com muitas colunas.
#
#  5. COLUNAS DE TEMPO PROTEGIDAS
#     As chaves temporais (KEY_ANO, KEY_MES, KEY_DIA, KEY_HORA) nunca entram
#     no rastreio de constantes e, portanto, nunca são removidas, mesmo que
#     tecnicamente apresentem baixa variabilidade em algum recorte.
#
#  6. MEDIÇÃO DE TEMPO PADRONIZADA (TimeTracker)
#     Todas as medições de tempo do script usam a classe TimeTracker (context
#     manager), substituindo blocos soltos de time.perf_counter() por um
#     padrão único e reutilizável, com opção de log automático.
#
#  7. ABERTURA DE PARQUET COM RETRY
#     _open_parquet_with_retry() é um context manager que tenta reabrir o
#     Parquet de entrada até 3 vezes com espera entre tentativas, mitigando
#     falhas transitórias de acesso ao Google Drive, e garante fechamento
#     seguro do recurso via `with`.
#
#  8. ESCRITA EM DISCO LOCAL COM CÓPIA FINAL
#     Os Parquets de saída são escritos primeiro em DIR_TMP (disco local,
#     mais rápido) e só depois copiados para o destino final no Drive.
#
#  9. REPRODUTIBILIDADE
#     O JSON de constantes inclui metadados completos (colunas totais,
#     colunas removidas/mantidas, colunas de tempo protegidas, linhas do
#     treino, nº de row groups), permitindo recarregar o FIT sem
#     reprocessamento via `reload_stats=True` e `ConstantFitter.from_json()`.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas auxiliares
#  pandas            1.5            Manipulação de DataFrames em lote
#  pyarrow           10.0           Leitura por row group, escrita Parquet
#  rich              13.0           Painéis, tabelas e progresso no terminal
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich pyarrow
#  (numpy e pandas já estão disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Entrada (Step 6):
#    .../6_tratamento_de_faltantes/
#    └── cenario_A_todos_anos/
#        ├── treino/<ESTRATEGIA>/TREINO_IMPUTADO.parquet
#        └── teste/<ESTRATEGIA>/TESTE_IMPUTADO.parquet
#
#  Saída (Step 7 — mesma estrutura espelhada):
#    .../7_tratamento_outliers/
#    └── cenario_A_todos_anos/
#        ├── treino/<ESTRATEGIA>/TREINO_CLEAN.parquet
#        ├── teste/<ESTRATEGIA>/TESTE_CLEAN.parquet
#        ├── estatisticas_fit/<ESTRATEGIA>/constant_cols.json
#        └── relatorios/relatorio_outliers.csv
#    relatorio_consolidado_outliers.csv
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich pyarrow
#
#  Célula 3 — Executar o script (após o Step 6 já ter sido executado):
#      exec(open('tratamento_outliers_tcc_v21.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import gc
import json
import shutil
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Dict, Generator, List, Optional, Set

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from rich.console import Console
from rich.panel import Panel
from rich.progress import (
    BarColumn,
    MofNCompleteColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)
from rich.table import Table
from rich.theme import Theme
from rich import box


# ==============================================================================
# 🎨 TEMA E CONSOLE
# ==============================================================================

_THEME = Theme({
    "info":      "bold cyan",
    "warning":   "bold yellow",
    "error":     "bold red",
    "success":   "bold green",
    "header":    "bold white on blue",
    "metric":    "bold magenta",
    "dim":       "dim white",
    "treino":    "bold steel_blue1",
    "teste":     "bold orange1",
    "cenario_a": "bold green",
})

console = Console(theme=_THEME)


# ==============================================================================
# ⚙️ CONFIGURAÇÕES GLOBAIS
# ==============================================================================

BASE_PATH = Path("/content/drive/MyDrive/TCC_PLD_Project")

# ── Input: saída da etapa de tratamento de faltantes (Step 6) ────────────────
INPUT_BASE = BASE_PATH / "09-ESCRITA_TCC/PARTES_TCC/codigos/dados/6_tratamento_de_faltantes"

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_BASE = BASE_PATH / "09-ESCRITA_TCC/PARTES_TCC/codigos/dados/7_tratamento_outliers"

DIR_TMP = Path("/tmp/outlier_engine_v2")

# ── Cenário herdado da etapa de tratamento de faltantes ──────────────────────
SCENARIOS: List[Dict] = [
    {
        "label": "cenario_A_todos_anos",
        "desc":  "FIT com todos os anos do treino",
    },
]

# ── Estratégias herdadas da etapa de tratamento de faltantes ────────────────
STRATEGIES: List[str] = [
    "HYBRID_AGGRESSIVE"
]

# ── Chaves de tempo (nunca removidas mesmo se constantes) ─────────────────────
TIME_COLS: List[str] = ["KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA"]

# ── Processamento ─────────────────────────────────────────────────────────────
DEFAULT_BATCH_SIZE = 15_000


# ==============================================================================
# ⏱️ UTILITÁRIOS
# ==============================================================================

def fmt_n(n: int) -> str:
    return f"{n:,}".replace(",", ".")


def bytes_to_mb(b: int) -> float:
    return round(b / 1024 ** 2, 2)


class TimeTracker:
    """
    Context manager padronizado para medição de tempo de execução.

    Substitui blocos espalhados de time.perf_counter() por um padrão
    coeso e reutilizável.

    Uso:
        with TimeTracker() as t:
            operacao_pesada()
        console.print(f"Concluído em {t.elapsed_s}s")

        # Com rótulo e log automático:
        with TimeTracker(label="FIT Cenário A", auto_log=True) as t:
            fitter.fit()
    """

    def __init__(self, label: str = "", auto_log: bool = False) -> None:
        self.label     = label
        self.auto_log  = auto_log
        self.elapsed_s: float = 0.0
        self._start:    float = 0.0

    def __enter__(self) -> "TimeTracker":
        self._start = time.perf_counter()
        return self

    def __exit__(self, *_) -> None:
        self.elapsed_s = round(time.perf_counter() - self._start, 2)
        if self.auto_log and self.label:
            console.print(
                f"   [dim]⏱  {self.label}:[/] [metric]{self.elapsed_s}s[/]\n"
            )

    @property
    def elapsed_str(self) -> str:
        return f"{self.elapsed_s:.2f}s"


@contextmanager
def _open_parquet_with_retry(
    path: Path, retries: int = 3, delay: float = 5.0
) -> Generator[pq.ParquetFile, None, None]:
    """
    Context manager que abre um ParquetFile com retentativas automáticas.
    Garante uso seguro via with e evita file handles órfãos.
    """
    last_exc: Exception = RuntimeError("Nenhuma tentativa realizada")
    pf: Optional[pq.ParquetFile] = None

    for attempt in range(1, retries + 1):
        try:
            pf = pq.ParquetFile(str(path))
            break
        except Exception as exc:
            last_exc = exc
            console.print(
                f"   [warning]⚠  Tentativa {attempt}/{retries} → {path.name}: {exc}. "
                f"Aguardando {delay}s...[/]"
            )
            time.sleep(delay)

    if pf is None:
        raise last_exc

    try:
        yield pf
    finally:
        # ParquetFile não possui .close() explícito, mas encapsular no
        # context manager protege contra expansões futuras da API PyArrow.
        del pf


def _safe_cast_table(table: pa.Table, schema: pa.Schema) -> pa.Table:
    """Alinha o schema de um batch ao schema esperado, com cast seguro."""
    arrays = []
    for field in schema:
        if field.name in table.schema.names:
            col = table.column(field.name)
            try:
                arrays.append(col.cast(field.type))
            except (pa.ArrowInvalid, pa.ArrowNotImplementedError):
                arrays.append(col)
        else:
            arrays.append(pa.array([None] * len(table), type=field.type))
    return pa.table(arrays, schema=schema)


# ==============================================================================
# 📐 FITTER DE CONSTANTES — RAM-safe + scan por row group
# ==============================================================================

class ConstantFitter:
    """
    Detecta colunas constantes APENAS no TREINO (anti-leakage).

    Estratégia — três ganhos sobre uma leitura coluna a coluna:

    1. Itera por ROW GROUP (não coluna a coluna):
       Em vez de abrir o arquivo uma vez por coluna do catálogo, faz N
       passagens onde N = número de row groups — tipicamente poucas
       dezenas.

    2. Lê APENAS as colunas ainda em rastreio a cada row group:
       Assim que uma coluna acumula >1 valor único (early-exit), ela sai do
       set de rastreio. Os row groups seguintes leem menos colunas → RAM
       cai progressivamente ao longo do scan.

    3. Nunca materializa o arquivo inteiro em memória:
       Cada row group é lido, processado e descartado antes do próximo.

    4. Medição de tempo via TimeTracker (context manager padronizado).
    """

    def __init__(self, treino_path: Path, batch_size: int = DEFAULT_BATCH_SIZE) -> None:
        self.treino_path    = treino_path
        self.batch_size     = batch_size
        self.constant_cols: Set[str]  = set()
        self.all_cols:      List[str] = []
        self._total_rows:   int = 0
        self._num_rg:       int = 0

    # ──────────────────────────────────────────────────────────────────────────
    def fit(self, label: str = "") -> "ConstantFitter":
        if not self.treino_path.exists():
            raise FileNotFoundError(f"Treino não encontrado: {self.treino_path}")

        console.print(
            Panel(
                "[treino]FIT — constantes detectadas APENAS no TREINO[/]\n"
                "[dim]A mesma lista será aplicada ao TESTE (anti-leakage)[/]\n"
                "[dim]Scan por row group + colunas filtradas + early-exit[/]",
                title=f"[header] FIT CONSTANTES — {label} [/]",
                border_style="steel_blue1",
                padding=(0, 2),
            )
        )

        meta             = pq.read_metadata(str(self.treino_path))
        schema           = pq.read_schema(str(self.treino_path))
        self._total_rows = meta.num_rows
        self._num_rg     = meta.num_row_groups
        self.all_cols    = schema.names

        console.print(
            f"   [info]Linhas no treino[/]: [metric]{fmt_n(self._total_rows)}[/]  |  "
            f"[info]Colunas totais[/]: [metric]{fmt_n(len(self.all_cols))}[/]  |  "
            f"[info]Row groups[/]: [metric]{self._num_rg}[/]\n"
        )

        # Colunas a rastrear — TIME_COLS nunca são removidas
        # tracking[col] = set de valores únicos vistos até agora (sem NaN)
        tracking: Dict[str, set] = {
            col: set()
            for col in self.all_cols
            if col not in TIME_COLS
        }

        with TimeTracker(label=f"FIT scan — {label}") as timer:
            with _open_parquet_with_retry(self.treino_path) as pf:
                with Progress(
                    SpinnerColumn(),
                    TextColumn("[treino]{task.description}[/]"),
                    BarColumn(),
                    TextColumn("[metric]{task.completed}/{task.total} row groups[/]"),
                    TextColumn("[dim]rastreando {task.fields[n_tracking]} cols[/]"),
                    TimeElapsedColumn(),
                    console=console,
                    transient=True,
                ) as progress:
                    task = progress.add_task(
                        "Detectando constantes...",
                        total=self._num_rg,
                        n_tracking=len(tracking),
                    )

                    for rg_idx in range(self._num_rg):
                        if not tracking:
                            progress.advance(task)
                            break

                        cols_to_read = list(tracking.keys())

                        try:
                            table_rg = pf.read_row_group(rg_idx, columns=cols_to_read)
                        except Exception as exc:
                            console.print(
                                f"   [warning]⚠  Row group {rg_idx}: {exc} — pulando[/]"
                            )
                            progress.advance(task)
                            continue

                        to_drop: List[str] = []

                        for col in cols_to_read:
                            if col not in table_rg.schema.names:
                                continue

                            arr   = table_rg.column(col)
                            valid = arr.drop_null()

                            if len(valid) == 0:
                                continue

                            chunk_uniques = set(valid.to_pandas().unique())
                            tracking[col].update(chunk_uniques)

                            if len(tracking[col]) > 1:
                                to_drop.append(col)

                        for col in to_drop:
                            del tracking[col]

                        del table_rg
                        gc.collect()

                        progress.update(task, advance=1, n_tracking=len(tracking))

        # O que permanece em tracking tem 0 ou 1 valor único → constante
        self.constant_cols = set(tracking.keys())

        console.print(
            f"   [dim]Scan concluído em [/][metric]{timer.elapsed_str}[/]"
            f"[dim] | {self._num_rg} row groups | "
            f"constantes encontradas: [/][metric]{len(self.constant_cols)}[/]\n"
        )

        self._print_fit_summary(label)
        return self

    # ──────────────────────────────────────────────────────────────────────────
    def _print_fit_summary(self, label: str = "") -> None:
        n_const = len(self.constant_cols)
        n_total = len(self.all_cols)
        n_kept  = n_total - n_const

        t = Table(
            title=f"Resultado do FIT de Constantes — {label}",
            border_style="steel_blue1", box=box.SIMPLE_HEAD, show_lines=True,
        )
        t.add_column("Métrica",  style="bold white", justify="left",  min_width=38)
        t.add_column("Valor",    style="metric",     justify="right", min_width=14)
        t.add_column("% Total",  style="dim",        justify="right", min_width=10)

        rows_data = [
            ("Colunas totais no treino",             n_total, "100.0%"),
            ("Colunas constantes (serão removidas)", n_const, f"{n_const/n_total*100:.1f}%"),
            ("Colunas mantidas",                     n_kept,  f"{n_kept/n_total*100:.1f}%"),
        ]
        colors = ["white", "red", "green"]
        for (lbl, val, pct), cor in zip(rows_data, colors):
            t.add_row(f"[{cor}]{lbl}[/{cor}]", f"{val:,}", pct)

        console.print(t)
        console.print(
            f"   [success]✔  FIT concluído:[/] "
            f"[metric]{n_const}[/] colunas constantes identificadas no treino\n"
        )

    # ──────────────────────────────────────────────────────────────────────────
    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "__meta__": {
                "fitter_version":        "2.1",
                "total_cols":            len(self.all_cols),
                "constant_cols":         len(self.constant_cols),
                "kept_cols":             len(self.all_cols) - len(self.constant_cols),
                "time_cols_protected":   TIME_COLS,
                "total_rows_treino":     self._total_rows,
                "num_row_groups":        self._num_rg,
            },
            "constant_cols": sorted(self.constant_cols),
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        console.print(f"   [success]✔  Lista de constantes salva:[/] [dim]{path}[/]\n")

    # ──────────────────────────────────────────────────────────────────────────
    @classmethod
    def from_json(cls, path: Path) -> "ConstantFitter":
        obj = cls.__new__(cls)
        obj.batch_size  = DEFAULT_BATCH_SIZE
        obj._total_rows = 0
        obj._num_rg     = 0
        obj.all_cols    = []
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        obj.constant_cols = set(raw.get("constant_cols", []))
        meta              = raw.get("__meta__", {})
        obj._total_rows   = meta.get("total_rows_treino", 0)
        obj._num_rg       = meta.get("num_row_groups", 0)
        console.print(
            f"   [info]Lista de constantes carregada:[/] [dim]{path}[/] "
            f"[metric]({len(obj.constant_cols):,} colunas)[/]\n"
        )
        return obj


# ==============================================================================
# 🔧 TRANSFORMER DE CONSTANTES — remove colunas usando lista do FIT
# ==============================================================================

class ConstantTransformer:
    """
    Remove do dataset as colunas identificadas como constantes no FIT do treino.
    Processa em batches de linhas para segurança de RAM.
    Medição de tempo via TimeTracker (context manager padronizado).
    """

    def __init__(
        self,
        constant_cols: Set[str],
        batch_size:    int = DEFAULT_BATCH_SIZE,
    ) -> None:
        self.constant_cols = constant_cols
        self.batch_size    = batch_size

    def transform(
        self,
        input_path:        Path,
        output_path_tmp:   Path,
        output_path_final: Path,
        split_label:       str = "TREINO",
    ) -> Dict:
        if not input_path.exists():
            raise FileNotFoundError(f"Input não encontrado: {input_path}")

        cor = "treino" if split_label == "TREINO" else "teste"

        schema_cols  = set(pq.read_schema(str(input_path)).names)
        cols_to_drop = self.constant_cols & schema_cols
        cols_to_keep = [
            c for c in pq.read_schema(str(input_path)).names
            if c not in cols_to_drop
        ]

        console.print(
            f"   [{cor}]{split_label}[/] → "
            f"removendo [metric]{len(cols_to_drop)}[/] colunas constantes | "
            f"mantendo [metric]{len(cols_to_keep)}[/] colunas\n"
        )

        writer:          Optional[pq.ParquetWriter] = None
        expected_schema: Optional[pa.Schema]        = None
        total_rows:      int = 0

        output_path_tmp.parent.mkdir(parents=True, exist_ok=True)

        with TimeTracker(label=f"TRANSFORM {split_label}") as timer:
            try:
                with _open_parquet_with_retry(input_path) as pf:
                    with Progress(
                        SpinnerColumn(),
                        TextColumn(f"[{cor}]{{task.description}}[/]"),
                        BarColumn(), MofNCompleteColumn(), TimeElapsedColumn(),
                        console=console, transient=True,
                    ) as progress:
                        task = progress.add_task(
                            f"Limpando constantes — {split_label}...", total=None
                        )

                        for batch in pf.iter_batches(batch_size=self.batch_size):
                            df = batch.to_pandas()
                            total_rows += len(df)

                            drop_here = [c for c in cols_to_drop if c in df.columns]
                            df = df.drop(columns=drop_here, errors="ignore")

                            if writer is None:
                                table_out       = pa.Table.from_pandas(df, preserve_index=False)
                                expected_schema = table_out.schema
                                writer = pq.ParquetWriter(
                                    str(output_path_tmp), expected_schema,
                                    compression="snappy"
                                )
                            else:
                                table_out = pa.Table.from_pandas(df, preserve_index=False)
                                table_out = _safe_cast_table(
                                    table_out, expected_schema  # type: ignore[arg-type]
                                )

                            writer.write_table(table_out)

                            del df, table_out
                            gc.collect()
                            progress.advance(task)

            finally:
                if writer is not None:
                    try:
                        writer.close()
                    except Exception as e:
                        console.print(f"   [warning]⚠  Erro ao fechar writer: {e}[/]")

        output_path_final.parent.mkdir(parents=True, exist_ok=True)
        console.print(f"   [dim]Copiando {output_path_tmp.name} → Drive...[/]")
        shutil.copy2(str(output_path_tmp), str(output_path_final))
        output_path_tmp.unlink(missing_ok=True)

        out_size_mb = bytes_to_mb(output_path_final.stat().st_size)

        result = {
            "split":        split_label,
            "rows":         total_rows,
            "cols_dropped": len(cols_to_drop),
            "cols_kept":    len(cols_to_keep),
            "elapsed_s":    timer.elapsed_s,
            "output_mb":    out_size_mb,
            "path":         str(output_path_final),
            "status":       "OK",
        }

        console.print(
            f"   [success]✔ {split_label}[/] "
            f"| {fmt_n(total_rows)} linhas "
            f"| [metric]{timer.elapsed_str}[/] "
            f"| [metric]{out_size_mb} MB[/] "
            f"| Removidas: [metric]{len(cols_to_drop)}[/] "
            f"| Mantidas: [metric]{len(cols_to_keep)}[/]\n"
        )
        return result


# ==============================================================================
# 🏗️ MOTOR PRINCIPAL
# ==============================================================================

class OutlierEngineV2:
    """
    Itera sobre o cenário e estratégias herdados da etapa de tratamento de
    faltantes, aplica remoção de constantes com anti-leakage (fit no treino,
    transform no teste) e espelha a estrutura de pastas em
    7_tratamento_outliers/.

    Características desta versão:
    - TimeTracker como context manager padronizado para toda medição de tempo.
    - _open_parquet_with_retry como context manager (@contextmanager).
    - Pipeline local, sem requisições HTTP externas.
    """

    def __init__(
        self,
        input_base:   Path                 = INPUT_BASE,
        output_base:  Path                 = OUTPUT_BASE,
        batch_size:   int                  = DEFAULT_BATCH_SIZE,
        scenarios:    Optional[List[Dict]] = None,
        strategies:   Optional[List[str]]  = None,
        reload_stats: bool                 = False,
    ) -> None:
        self.input_base   = Path(input_base)
        self.output_base  = Path(output_base)
        self.batch_size   = batch_size
        self.scenarios    = scenarios or SCENARIOS
        self.strategies   = strategies or STRATEGIES
        self.reload_stats = reload_stats
        self._all_results: List[Dict] = []

        DIR_TMP.mkdir(parents=True, exist_ok=True)
        self.output_base.mkdir(parents=True, exist_ok=True)

    # ──────────────────────────────────────────────────────────────────────────
    def run(self) -> None:
        self._print_banner()

        with TimeTracker(label="Pipeline completo") as total_timer:
            for scenario in self.scenarios:
                self._run_scenario(scenario)

        self._print_final_report(total_timer.elapsed_s)
        self._salvar_relatorio_consolidado()

    # ──────────────────────────────────────────────────────────────────────────
    def _run_scenario(self, scenario: Dict) -> None:
        label  = scenario["label"]
        desc   = scenario["desc"]
        cor    = "cenario_a"
        titulo = "CENÁRIO A — TODOS OS ANOS"

        console.print()
        console.rule(f"[{cor}]{'=' * 20}  {titulo}  {'=' * 20}[/]")
        console.print(
            Panel(
                f"[{cor}]{desc}[/]\n"
                f"[dim]Input : {self.input_base / label}[/]\n"
                f"[dim]Output: {self.output_base / label}[/]",
                title=f"[header] {titulo} [/]",
                border_style="green",
                padding=(0, 2),
            )
        )

        for strategy in self.strategies:
            console.rule(f"[header] {label} | ESTRATÉGIA: {strategy} [/]")

            # ── Caminhos de entrada (Step 6) ──────────────────────────────────
            treino_in = self.input_base / label / "treino" / strategy / "TREINO_IMPUTADO.parquet"
            teste_in  = self.input_base / label / "teste"  / strategy / "TESTE_IMPUTADO.parquet"

            # ── Caminhos de saída (Step 7) ────────────────────────────────────
            treino_out_final = self.output_base / label / "treino" / strategy / "TREINO_CLEAN.parquet"
            teste_out_final  = self.output_base / label / "teste"  / strategy / "TESTE_CLEAN.parquet"
            treino_out_tmp   = DIR_TMP / f"{label}_{strategy}_TREINO.parquet"
            teste_out_tmp    = DIR_TMP / f"{label}_{strategy}_TESTE.parquet"

            # ── JSON com lista de constantes ──────────────────────────────────
            stats_dir  = self.output_base / label / "estatisticas_fit" / strategy
            stats_path = stats_dir / "constant_cols.json"

            # ── FIT no treino ──────────────────────────────────────────────────
            if self.reload_stats and stats_path.exists():
                console.print(
                    Panel(
                        f"[info]Reutilizando lista de constantes:[/]\n[dim]{stats_path}[/]",
                        title="[header] FIT (RELOAD) [/]",
                        border_style="steel_blue1",
                    )
                )
                fitter = ConstantFitter.from_json(stats_path)
            else:
                if not treino_in.exists():
                    console.print(
                        f"   [error]✘  Treino não encontrado: {treino_in}[/]\n"
                        f"   [dim]Verifique se a etapa de tratamento de faltantes "
                        f"(Step 6) foi executada para este cenário/estratégia.[/]\n"
                    )
                    self._all_results.append(
                        self._erro("TREINO", strategy, label, "arquivo não encontrado")
                    )
                    self._all_results.append(
                        self._erro("TESTE", strategy, label, "arquivo não encontrado")
                    )
                    continue

                fitter = ConstantFitter(treino_in, self.batch_size).fit(
                    label=f"{label} | {strategy}"
                )
                fitter.save(stats_path)

            transformer = ConstantTransformer(fitter.constant_cols, self.batch_size)

            # ── TRANSFORM treino ───────────────────────────────────────────────
            try:
                r = transformer.transform(
                    input_path        = treino_in,
                    output_path_tmp   = treino_out_tmp,
                    output_path_final = treino_out_final,
                    split_label       = "TREINO",
                )
                r.update({"cenario": label, "strategy": strategy})
                self._all_results.append(r)
            except Exception:
                console.print_exception()
                self._all_results.append(self._erro("TREINO", strategy, label))

            # ── TRANSFORM teste (lista do treino — anti-leakage) ──────────────
            try:
                r = transformer.transform(
                    input_path        = teste_in,
                    output_path_tmp   = teste_out_tmp,
                    output_path_final = teste_out_final,
                    split_label       = "TESTE",
                )
                r.update({"cenario": label, "strategy": strategy})
                self._all_results.append(r)
            except Exception:
                console.print_exception()
                self._all_results.append(self._erro("TESTE", strategy, label))

        # ── Salva relatório do cenário ─────────────────────────────────────────
        self._salvar_relatorio_cenario(
            [r for r in self._all_results if r.get("cenario") == label],
            self.output_base / label / "relatorios",
        )

    # ──────────────────────────────────────────────────────────────────────────
    @staticmethod
    def _erro(
        split:    str,
        strategy: str,
        cenario:  str,
        motivo:   str = "exceção",
    ) -> Dict:
        return {
            "cenario":      cenario,
            "split":        split,
            "strategy":     strategy,
            "rows":         0,
            "cols_dropped": 0,
            "cols_kept":    0,
            "elapsed_s":    0,
            "output_mb":    0,
            "path":         "—",
            "status":       f"ERRO ({motivo})",
        }

    # ──────────────────────────────────────────────────────────────────────────
    def _print_banner(self) -> None:
        cenarios_str = "\n".join(
            f"  [cenario_a]A[/cenario_a]"
            f"  [dim]{s['label']:35s} | {s['desc']}[/dim]"
            for s in self.scenarios
        )
        console.print()
        console.print(
            Panel(
                "[bold white]MOTOR DE TRATAMENTO DE OUTLIERS v2.1 — REMOÇÃO DE CONSTANTES | CENÁRIO ÚNICO[/]\n"
                "[dim]FIT por row group (PyArrow) | early-exit por coluna | RAM-safe[/]\n"
                "[dim]TimeTracker context manager[/]\n\n"
                f"[dim]Input  (Step 6): {self.input_base}[/]\n"
                f"[dim]Output (Step 7): {self.output_base}[/]\n\n"
                f"[bold white]Cenário:[/]\n{cenarios_str}\n\n"
                f"[dim]Estratégias: {', '.join(self.strategies)}[/]",
                title="[header]  TCC PLD HORÁRIO — STEP 7: TRATAMENTO DE OUTLIERS  [/]",
                border_style="blue",
                padding=(1, 4),
            )
        )
        console.print(
            Panel(
                "[warning]GARANTIA ANTI-LEAKAGE:[/]\n\n"
                "  [treino]TREINO[/]  →  FIT (constantes detectadas aqui)  +  TRANSFORM\n"
                "  [teste]TESTE[/]   →  TRANSFORM apenas (lista do treino aplicada)\n\n"
                "  [dim]O TESTE nunca influencia quais colunas são removidas.[/]\n"
                "  [dim]Lista auditável em: "
                "<cenario>/estatisticas_fit/<estrategia>/constant_cols.json[/]",
                title="[header] ARQUITETURA ANTI-LEAKAGE [/]",
                border_style="yellow",
                padding=(0, 2),
            )
        )

    # ──────────────────────────────────────────────────────────────────────────
    def _print_final_report(self, elapsed_total: float) -> None:
        t = Table(
            title="RELATÓRIO CONSOLIDADO — STEP 7: TRATAMENTO DE OUTLIERS",
            border_style="blue", box=box.DOUBLE_EDGE, show_lines=True, min_width=150,
        )
        t.add_column("Cenário",    style="bold",      justify="left",   min_width=28)
        t.add_column("Split",      style="bold",      justify="center", min_width=8)
        t.add_column("Estratégia", style="bold cyan", justify="left",   min_width=22)
        t.add_column("Registros",  style="metric",    justify="right",  min_width=12)
        t.add_column("Removidas",  style="red",       justify="right",  min_width=10)
        t.add_column("Mantidas",   style="green",     justify="right",  min_width=10)
        t.add_column("Saída (MB)", style="green",     justify="right",  min_width=10)
        t.add_column("Tempo (s)",  style="white",     justify="right",  min_width=9)
        t.add_column("Status",     style="bold",      justify="center", min_width=7)

        for r in self._all_results:
            cor_c  = "cenario_a"
            cor_s  = "treino" if r["split"] == "TREINO" else "teste"
            status = "[green]OK[/]" if r["status"] == "OK" else f"[red]{r['status']}[/]"
            t.add_row(
                f"[{cor_c}]{r.get('cenario', '—')}[/{cor_c}]",
                f"[{cor_s}]{r['split']}[/{cor_s}]",
                r.get("strategy", "—"),
                fmt_n(r["rows"]),
                f"{r['cols_dropped']:,}",
                f"{r['cols_kept']:,}",
                f"{r['output_mb']}",
                f"{r['elapsed_s']}",
                status,
            )

        console.print(t)

        n_ok  = sum(1 for r in self._all_results if r["status"] == "OK")
        n_err = len(self._all_results) - n_ok

        console.print(
            Panel(
                f"[success]PIPELINE COMPLETO EM {elapsed_total:.2f}s[/]\n\n"
                f"   [dim]Cenários    : {len(self.scenarios)} (A)[/]\n"
                f"   [dim]Estratégias : {len(self.strategies)}[/]\n"
                f"   [dim]Splits      : TREINO + TESTE[/]\n"
                f"   [dim]Arquivos OK : {n_ok}  |  Erros: {n_err}[/]\n"
                f"   [dim]Output base : {self.output_base}[/]",
                title="[header] RESUMO FINAL — STEP 7 [/]",
                border_style="green",
                padding=(1, 4),
            )
        )
        console.print(
            Panel(
                "  OK  Constantes detectadas APENAS no treino (fit)\n"
                "  OK  Teste recebeu TRANSFORM com lista do treino (anti-leakage)\n"
                "  OK  Colunas de tempo protegidas (nunca removidas)\n"
                "  OK  FIT: scan por row group + early-exit (RAM-safe)\n"
                "  OK  TimeTracker como context manager padronizado\n"
                "  OK  Lista auditável por cenário e estratégia em constant_cols.json\n\n"
                "  [dim]Para reexecutar sem refazer o fit, use reload_stats=True[/]",
                title="[header] AVISO METODOLÓGICO — TCC [/]",
                border_style="yellow",
                padding=(0, 2),
            )
        )

    # ──────────────────────────────────────────────────────────────────────────
    def _salvar_relatorio_cenario(self, results: List[Dict], dir_out: Path) -> None:
        if not results:
            return
        dir_out.mkdir(parents=True, exist_ok=True)
        path = dir_out / "relatorio_outliers.csv"
        pd.DataFrame(results).to_csv(path, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório do cenário salvo:[/] [dim]{path}[/]\n")

    def _salvar_relatorio_consolidado(self) -> None:
        path = self.output_base / "relatorio_consolidado_outliers.csv"
        pd.DataFrame(self._all_results).to_csv(path, index=False, encoding="utf-8-sig")
        console.print(
            f"   [success]✔  Relatório consolidado salvo:[/] [dim]{path}[/]\n"
        )


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    engine = OutlierEngineV2(
        input_base   = INPUT_BASE,
        output_base  = OUTPUT_BASE,
        batch_size   = DEFAULT_BATCH_SIZE,
        scenarios    = SCENARIOS,
        strategies   = STRATEGIES,
        reload_stats = False,   # True → pula o fit e reutiliza constant_cols.json salvos
    )
    engine.run()